<a href="https://colab.research.google.com/github/brahime2000-hue/contract-app/blob/main/Prompt_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

# =========================================================
# GREEN + WHITE + BLACK UI
# =========================================================
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');

:root{
    --bg:#000000;
    --panel:#0d0d0d;
    --panel2:#151515;
    --line:#2a2a2a;

    --text:#f5f5f5;
    --muted:#a5a5a5;

    --accent:#00c853;
    --accent-soft:#39e27d;

    --radius:14px;
}

body, .jp-Notebook, .jp-Cell, .jp-OutputArea-output{
    background:var(--bg)!important;
    color:var(--text)!important;
    font-family:'Inter',sans-serif!important;
}

/* main inputs */
.widget-text input,
.widget-textarea textarea{
    background:var(--panel)!important;
    color:var(--text)!important;
    border:1px solid var(--line)!important;
    border-radius:12px!important;
    padding:10px 12px!important;
    font-size:13px!important;
    box-shadow:none!important;
}

.widget-text input:focus,
.widget-textarea textarea:focus{
    border-color:var(--accent)!important;
    box-shadow:0 0 0 2px rgba(0,200,83,.14)!important;
}

/* buttons */
.widget-button button{
    border:none!important;
    border-radius:10px!important;
    font-size:13px!important;
    font-weight:700!important;
    padding:9px 16px!important;
    box-shadow:none!important;
    transition:all .18s ease!important;
}

.widget-button button:hover{
    filter:brightness(1.06)!important;
    transform:translateY(-1px)!important;
}

.widget-button button:active{
    transform:translateY(0)!important;
}

/* primary/generate button */
.widget-button button.mod-primary,
.widget-button button.jupyter-widgets.jupyter-button.widget-button.mod-primary{
    background:linear-gradient(135deg,var(--accent),var(--accent-soft))!important;
    color:#000000!important;
}

/* tabs */
.widget-tab .p-TabBar-tab{
    background:var(--panel2)!important;
    color:var(--muted)!important;
    border:1px solid var(--line)!important;
    border-bottom:none!important;
    border-radius:10px 10px 0 0!important;
    padding:10px 18px!important;
    font-size:13px!important;
    font-weight:700!important;
}

.widget-tab .p-TabBar-tab:hover{
    color:var(--text)!important;
}

.widget-tab .p-TabBar-tab.p-mod-current{
    background:var(--panel)!important;
    color:var(--text)!important;
    box-shadow:inset 0 2px 0 var(--accent)!important;
}

.widget-tab > .p-Widget{
    background:var(--panel)!important;
    border:1px solid var(--line)!important;
    border-radius:0 0 14px 14px!important;
    padding:14px!important;
}

/* labels */
.ui-title{
    font-size:26px;
    font-weight:800;
    color:var(--text);
    margin:0 0 14px 0;
    letter-spacing:.2px;
}

.ui-title span{
    color:var(--accent);
}

.ui-label{
    font-size:11px;
    color:var(--muted);
    margin:14px 0 6px 2px;
    text-transform:uppercase;
    letter-spacing:1.2px;
    font-weight:700;
}

.small-label{
    font-size:11px;
    color:var(--muted);
    margin:0 0 6px 2px;
    text-transform:uppercase;
    letter-spacing:1.2px;
    font-weight:700;
}

.status-text{
    font-size:12px;
    color:var(--accent-soft);
    margin-top:8px;
}

/* prompt text areas */
.prompt-area textarea{
    font-family:Consolas, monospace!important;
    line-height:1.4!important;
}
</style>
"""))

# =========================================================
# HELPERS
# =========================================================
def safe_value(v, fallback):
    v = (v or "").strip()
    return v if v else fallback

def copy_to_clipboard(text):
    js_safe = json.dumps(text)
    display(Javascript(f"navigator.clipboard.writeText({js_safe});"))

def flash(btn):
    old_desc = btn.description
    old_style = btn.button_style
    btn.description = "Copied"
    btn.button_style = "success"
    import threading
    def reset():
        btn.description = old_desc
        btn.button_style = old_style
    threading.Timer(1.2, reset).start()

def clear_outputs():
    prompt1_area.value = ""
    prompt2_area.value = ""
    prompt3_area.value = ""
    prompt4_area.value = ""

# =========================================================
# INPUTS
# =========================================================
TEST1FORPROMPT1 = widgets.Textarea(
    placeholder='Paste reference script...',
    layout=widgets.Layout(width='100%', height='240px')
)

TEST1FORPROMPT2 = widgets.Text(
    placeholder='Write video title...',
    layout=widgets.Layout(width='100%')
)

TEST1FORPROMPT3 = widgets.Text(
    value='3000',
    placeholder='3000',
    layout=widgets.Layout(width='180px')
)

TEST2FORPROMPT3 = widgets.Text(
    value='English',
    placeholder='English',
    layout=widgets.Layout(width='180px')
)

generate_btn = widgets.Button(
    description="Generate",
    button_style='primary',
    layout=widgets.Layout(width='160px', height='42px')
)

status_bar = widgets.HTML("<div class='status-text'>Ready</div>")

# =========================================================
# PROMPTS
# =========================================================
PROMPT_1 = """RESPONSE STYLE
- Short paragraphs + bullet points only.
- No long essays.
- Each subsection: max 6–8 bullets.
- Concrete, prescriptive.

TASK
Analyze ONLY Reference Script 1 (ignore timestamps) and extract its style/structure/rhythm DNA.
Output a reusable "Custom Style & Structure Guide" enabling ≥80–90% perceived similarity.
Do NOT: write a new story, plot-summarize for its own sake, ask questions.

REFERENCE SCRIPT 1
[REFERENCE_SCRIPT_START]
[TEST1FORPROMPT1]
[REFERENCE_SCRIPT_END]
"""

PROMPT_2 = """RESPONSE STYLE
Short paragraphs + bullets. Compact, practical.

LANGUAGE
Use [TARGET_LANGUAGE]. If [TARGET_LANGUAGE]="auto", use the user's/main language only. Do not switch.

INPUTS
New video title: [TEST1FORPROMPT2]
Target total word count: [TEST1FORPROMPT3]
"""

PROMPT_3 = """OUTPUT FORMAT
- Output ONLY narrative script text + the fixed English labels below.
- No headings, bullets, or "Act/Chapter" labels.
- Start immediately with story text (no meta/openers like "Here is…").

FIXED ENGLISH LINES
Hook variant 1
Hook variant 2
Hook variant 3
Section [BLOCK_ID] of [SECTIONS_COUNT]
Story Complete

Input: [BLOCK_ID] = HOOKS
Target total word count: [TEST1FORPROMPT3]

LANGUAGE
- Input: [TARGET_LANGUAGE] = [TEST2FORPROMPT3]
- If missing/empty/undefined/"auto": set [TARGET_LANGUAGE]="English".
- If explicitly set: write ALL story text (hooks/sections) in exactly [TARGET_LANGUAGE].
- Never ask the user about language.

ROLE
You are the SAME model that made: Prompt 1 Style Guide + Prompt 2 Outline/Section Plan.
Use them as the ONLY authority. Do NOT re-analyze reference. Do NOT ask for any inputs.
Assume TOTAL_USER, BODY_USER, SECTIONS_COUNT, and each section's Sk_TARGET exist from Prompt 2.

DEFINITIONS (HARD)
- Command "HOOKS" means: generate 3 hook variants ONLY (no sections).
- Command "block k" / "section k" / "BLOCK_ID=k" / "k" means: generate Section k ONLY.
- Command "block a-b" / "section a-b" / "BLOCK_ID=a-b" / "a-b" means: generate Sections a..b ONLY (range mode).
- Hook variants (1/2/3) are NOT sections and MUST NEVER be treated as section numbers.

STATELESS NUMBERING (HARD)
- Do NOT track progress or auto-advance section numbers.
- The ONLY source of truth is the last user command message.

CONTROL INPUT PARSING (HARD)
Treat the LAST user message as the control command.
IMPORTANT: If it contains extra text, ignore everything except a valid control pattern below.

1) RANGE MODE (highest priority)
- If the last message contains a valid range command:
  "block a-b" OR "section a-b" OR "BLOCK_ID=a-b" OR just "a-b"
  where a and b are integers, a<=b, and both in 1..[SECTIONS_COUNT],
  then set START=a and END=b → RANGE MODE.

2) SECTION MODE
- Else, if the last message contains a valid section command:
  "block k" OR "section k" OR "BLOCK_ID=k"
  where k is an integer in 1..[SECTIONS_COUNT],
  then set K=k → SECTION MODE.
- Else, if the entire last message is exactly an integer k in 1..[SECTIONS_COUNT],
  then set K=k → SECTION MODE.

3) HOOKS MODE
- Else, if the last message contains the standalone token "HOOKS" (case-insensitive),
  set MODE=HOOKS.

4) DEFAULT (NO QUESTIONS)
- If none match, default to HOOKS MODE.

ABSOLUTE SINGLE-MODE RULE (HARD)
In ONE answer:
- HOOKS MODE: output exactly 3 hooks and STOP.
- SECTION MODE: output exactly 1 section and STOP.
- RANGE MODE: output sections START..END only (no hooks) and STOP.
No explanations, no questions, no onboarding, no system talk.
"""

PROMPT_4 = """PROMPT 4 — STRUCTURAL STYLE EVALUATOR & REWRITER

RESPONSE STYLE
- Diagnostic: short paragraphs + bullets
- Rewrite: ONLY narrative script text + fixed English labels/questions (no bullets/headings/meta).
- Post-analysis: short paragraphs + bullets.

LANGUAGE CONTROL
Use the SAME dominant language as the current draft produced by Prompt 3 (detect once; never switch).
ONLY allowed English lines:
- "Should I rewrite the story to increase structural similarity to the reference? (Yes/No)"
- "Continue with Section k of N? (Yes/No)"
- "Would you like me to analyze the rewritten story, explain what I changed, and estimate the new structural similarity to the reference? (Yes/No)"
- Hook variant 1/2/3
- Section [BLOCK_ID] of [SECTIONS_COUNT]
- Story Complete

INPUTS
Target total word count: [TEST1FORPROMPT3]

ROLE
You are the SAME model that made Prompt 1–3. Use the existing guide/outline/section plan; do NOT re-invent style.

CURRENT DRAFT (internal)
Find the most recent Prompt 3 outputs in this chat:
- 3 hooks labeled Hook variant 1/2/3
- Sections labeled "Section k of N" (ordered by k)
Reconstruct full draft = hooks + all available sections.
If none found: output a short message (story language) saying Prompt 3 must be run first, then STOP.
"""

# =========================================================
# BUILD PROMPTS
# =========================================================
def build_prompts():
    words = safe_value(TEST1FORPROMPT3.value, "3000")
    lang  = safe_value(TEST2FORPROMPT3.value, "English")
    title = safe_value(TEST1FORPROMPT2.value, "")
    ref   = TEST1FORPROMPT1.value

    p1 = PROMPT_1.replace("[TEST1FORPROMPT1]", ref)

    p2 = (PROMPT_2
          .replace("[TARGET_LANGUAGE]", lang)
          .replace("[TEST1FORPROMPT2]", title)
          .replace("[TEST1FORPROMPT3]", words))

    p3 = (PROMPT_3
          .replace("[TEST1FORPROMPT3]", words)
          .replace("[TEST2FORPROMPT3]", lang))

    p4 = PROMPT_4.replace("[TEST1FORPROMPT3]", words)

    return {
        "Prompt 1": p1,
        "Prompt 2": p2,
        "Prompt 3": p3,
        "Prompt 4": p4
    }

# =========================================================
# OUTPUT AREAS
# =========================================================
PROMPT_HEIGHT = "390px"

prompt1_area = widgets.Textarea(
    layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT),
    value=""
)
prompt2_area = widgets.Textarea(
    layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT),
    value=""
)
prompt3_area = widgets.Textarea(
    layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT),
    value=""
)
prompt4_area = widgets.Textarea(
    layout=widgets.Layout(width='100%', height=PROMPT_HEIGHT),
    value=""
)

prompt1_area.add_class("prompt-area")
prompt2_area.add_class("prompt-area")
prompt3_area.add_class("prompt-area")
prompt4_area.add_class("prompt-area")

copy_btn_1 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_2 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_3 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))
copy_btn_4 = widgets.Button(description="Copy Prompt", button_style='', layout=widgets.Layout(width='130px'))

def make_tab(area, btn):
    return widgets.VBox(
        [
            widgets.HBox([btn], layout=widgets.Layout(justify_content='flex-end')),
            area
        ],
        layout=widgets.Layout(width='100%', height='450px')
    )

tabs = widgets.Tab(
    children=[
        make_tab(prompt1_area, copy_btn_1),
        make_tab(prompt2_area, copy_btn_2),
        make_tab(prompt3_area, copy_btn_3),
        make_tab(prompt4_area, copy_btn_4),
    ],
    layout=widgets.Layout(width='100%')
)

for i, name in enumerate(["Prompt 1", "Prompt 2", "Prompt 3", "Prompt 4"]):
    tabs.set_title(i, name)

# =========================================================
# ACTIONS
# =========================================================
def update_outputs():
    prompts = build_prompts()
    prompt1_area.value = prompts["Prompt 1"]
    prompt2_area.value = prompts["Prompt 2"]
    prompt3_area.value = prompts["Prompt 3"]
    prompt4_area.value = prompts["Prompt 4"]

def on_generate(_):
    update_outputs()
    status_bar.value = "<div class='status-text'>Generated</div>"

def copy_prompt(area, btn, name):
    if not area.value.strip():
        status_bar.value = f"<div class='status-text'>{name} is empty</div>"
        return
    copy_to_clipboard(area.value)
    flash(btn)
    status_bar.value = f"<div class='status-text'>{name} copied</div>"

generate_btn.on_click(on_generate)
copy_btn_1.on_click(lambda b: copy_prompt(prompt1_area, copy_btn_1, "Prompt 1"))
copy_btn_2.on_click(lambda b: copy_prompt(prompt2_area, copy_btn_2, "Prompt 2"))
copy_btn_3.on_click(lambda b: copy_prompt(prompt3_area, copy_btn_3, "Prompt 3"))
copy_btn_4.on_click(lambda b: copy_prompt(prompt4_area, copy_btn_4, "Prompt 4"))

# =========================================================
# SETTING BOX
# =========================================================
def setting_box(label, widget):
    return widgets.VBox(
        [
            widgets.HTML(f"<div class='small-label'>{label}</div>"),
            widget
        ],
        layout=widgets.Layout(width='190px')
    )

# =========================================================
# LAYOUT
# =========================================================
display(HTML("<div class='ui-title'>Prompt <span>Builder</span></div>"))

display(widgets.HTML("<div class='ui-label'>Reference Script</div>"))
display(TEST1FORPROMPT1)

display(widgets.HTML("<div class='ui-label'>Video Title</div>"))
display(TEST1FORPROMPT2)

display(widgets.HTML("<div class='ui-label'>Settings</div>"))
display(widgets.HBox(
    [
        setting_box("Target Words", TEST1FORPROMPT3),
        setting_box("Language", TEST2FORPROMPT3),
    ],
    layout=widgets.Layout(gap='14px', flex_flow='row wrap')
))

display(widgets.HTML("<div style='height:12px'></div>"))
display(generate_btn)
display(status_bar)

display(widgets.HTML("<div style='height:18px'></div>"))
display(tabs)

# =========================================================
# INIT
# =========================================================
clear_outputs()
status_bar.value = "<div class='status-text'>Ready</div>"

HTML(value="<div class='ui-label'>Reference Script</div>")

Textarea(value='', layout=Layout(height='240px', width='100%'), placeholder='Paste reference script...')

HTML(value="<div class='ui-label'>Video Title</div>")

Text(value='', layout=Layout(width='100%'), placeholder='Write video title...')

HTML(value="<div class='ui-label'>Settings</div>")

HTML(value="<div style='height:12px'></div>")

Button(button_style='primary', description='Generate', layout=Layout(height='42px', width='160px'), style=Butt…

HTML(value="<div class='status-text'>Ready</div>")

HTML(value="<div style='height:18px'></div>")